<a href="https://colab.research.google.com/github/veapasichnyk/HomeWorksDataLovesAcademy/blob/main/%D0%92%D0%B8%D0%BA%D0%BE%D1%80%D0%B8%D1%81%D1%82%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BF%D1%80%D0%BE%D0%BC%D0%BF%D1%82%D1%96%D0%B2_%D1%96_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%96%D0%B2_%D0%B2_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [1]:
!pip install -q langchain langchain-core langchain-community pydantic requests==2.32.4 feedparser
!pip install -q langchain-huggingface

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00


In [2]:
import json, getpass

hf_token = getpass.getpass("Введи свій HuggingFace токен (hf_...): ")

with open("creds.json", "w") as f:
    json.dump({"hf_api_token": hf_token}, f)

print("creds.json створено для HuggingFace")

Введи свій HuggingFace токен (hf_...): ··········
creds.json створено для HuggingFace


In [3]:
#Код для виклику LLM (Mistral)
from pathlib import Path
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Нові класи з langchain-huggingface
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

def load_creds(path="creds.json") -> dict:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError("Немає creds.json.")
    return json.loads(p.read_text(encoding="utf-8"))

def get_llm(temperature: float = 0.3):
    creds = load_creds()
    token = creds.get("hf_api_token")
    if not token:
        raise ValueError("У creds.json немає 'hf_api_token'")

    # HuggingFaceEndpoint - це LLM; обгортаємо у ChatHuggingFace для чат-пайплайна
    base_llm = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.3",
        task="text-generation",
        huggingfacehub_api_token=token,
        temperature=temperature,
        max_new_tokens=80,                    # щоб тримати лаконічність
        do_sample=True
    )
    return ChatHuggingFace(llm=base_llm)

PROMPT = ChatPromptTemplate.from_template("""
You are a concise expert explainer.
Topic: "{topic}"
Write in language: "{language}".
Answer MUST be <= 200 characters. Be ultra-brief.
Include: definition, key advantages, current research areas.
Avoid bullet points; one tight sentence with commas.
""".strip())

def ask_llm(topic: str="Квантові обчислення",
            language: Literal["uk","en"]="uk",
            temperature: float=0.3) -> str:
    chat = get_llm(temperature=temperature)
    chain = PROMPT | chat | StrOutputParser()
    return chain.invoke({"topic": topic, "language": language})

In [4]:
#Виклик українською
print(ask_llm(topic="Квантові обчислення", language="uk", temperature=0.3))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 "Квантові обчислення" - це обчислення, які використовують квантову механіку для обробки інформації. Вони мають потенціал досягти експоненціально швидших швидкостей, ніж класичні обчислення, завдяки квантовому паралелізму. Дослідження на сьогоднішній день зосереджені на створенні надійних квантових бітів, квантових алгоритмах оптимізації та квантових комп'ютерах з великою кількістю кварків.


In [5]:
#Call English
print(ask_llm(topic="Quantum computing", language="en", temperature=0.3))

 Quantum computing is a technology that utilizes quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. Key advantages include potential for exponentially faster solution to certain complex problems, like factoring large numbers or simulating quantum systems. Current research areas focus on error correction, scalability, and practical applications in fields like cryptography, drug discovery, and optimization.


**Пояснення**

Обрала temperature=0.3, бо потрібна лаконічна, фактична відповідь із мінімумом “креативних” відхилень. Низька температура зменшує варіативність, підвищує стабільність і чіткість.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [6]:
from pathlib import Path
import json
from typing import Literal

from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage

# Параметризований промпт
TEMPLATE = (
    'You are a concise expert explainer. '
    'Topic: "{topic}" '
    'Write in language: "{language}". '
    'Answer MUST be <= 200 characters. Be ultra-brief. '
    'Include: definition, key advantages, current research areas. '
    'Avoid bullet points; one tight sentence with commas.'
)
PROMPT_TMPL = PromptTemplate.from_template(TEMPLATE)
print("PromptTemplate готовий.")

PromptTemplate готовий.


In [7]:
#Завантаження токену та ініціалізація моделі
def load_creds(path="creds.json") -> dict:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError("Немає creds.json. Запустіть попередню клітинку та вставте HF токен.")
    return json.loads(p.read_text(encoding="utf-8"))

def get_llm(temperature: float = 0.3):
    creds = load_creds()
    token = creds.get("hf_api_token")
    if not token or not token.startswith("hf_"):
        raise ValueError("У creds.json немає коректного 'hf_api_token'. Перевірте попередню клітинку.")
    base_llm = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.3",
        task="text-generation",
        huggingfacehub_api_token=token,
        temperature=temperature,
        max_new_tokens=80,
        do_sample=True,
    )
    return ChatHuggingFace(llm=base_llm)

print("Функції завантажено.")

Функції завантажено.


In [8]:
#Формує промпт і викликає модель
def ask_llm(topic: str, language: Literal["uk","en"]="uk", temperature: float=0.3) -> str:
    prompt_text = PROMPT_TMPL.format(topic=topic, language=language)
    chat = get_llm(temperature=temperature)
    resp = chat.invoke([HumanMessage(content=prompt_text)])
    text = resp.content if hasattr(resp, "content") else str(resp)
    return text
print("ask_llm готова.")

ask_llm готова.


In [9]:
#Запуск на трьох темах
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI",
]

print("=== РЕЗУЛЬТАТИ МОДЕЛІ ===")
for t in topics:
    out = ask_llm(t, language="uk", temperature=0.3)
    print(f"\nТема: {t}\nВідповідь: {out}")

=== РЕЗУЛЬТАТИ МОДЕЛІ ===

Тема: Баєсівські методи в машинному навчанні
Відповідь:  Баєсівські методи - це статистичні методи, які використовують апріорні знання для передбачення ймовірності результату. Вони мають високу ефективність у випадку малої кількості даних та можуть бути застосовані для багатокласної класифікації. Поточні дослідження зосереджені на оптимізації алгоритмів та розширенні застосування в різних галузях, зокрема, в медицині та фінансах.

Тема: Трансформери в машинному навчанні
Відповідь:  Трансформери в машинному навчанні - це модель, яка перетворює вхідні дані в іншу форму, зменшуючи складність навчання та збільшуючи швидкість обробки. Головні переваги - зменшення розмірності даних, зниження вимог до обчислювальної потужності та покращення результатів навчання. Поточні дослідження стосуються покращення алгоритмів та застосування трансформерів у різних галузях, зокрема, обробці природної мови та комп'ютерному зору.

Тема: Explainable AI
Відповідь:  Explainable AI (X



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [9]:
!pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.3 MB/s eta 0:00:00


In [10]:
!pip install -q -U duckduckgo-search

In [30]:
# Crossref Tool
import json, re, html, requests
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from datetime import date
from langchain_core.tools import StructuredTool

CROSSREF_ENDPOINT = "https://api.crossref.org/works"

class CrossrefSearchInput(BaseModel):
    topic: str = Field(..., description="Тема пошуку")
    rows: int = Field(5, ge=1, le=50)
    from_date: Optional[str] = None
    until_date: Optional[str] = None
    require_abstract: bool = True

def _format_authors(auth_list):
    if not isinstance(auth_list, list):
        return []
    out = []
    for a in auth_list:
        given = (a.get("given") or "").strip()
        family = (a.get("family") or "").strip()
        name = " ".join([given, family]).strip() or (a.get("name") or "").strip()
        if not name:
            continue
        bad = ["department","faculty","university","institute","school","center","centre","laboratory"]
        if any(tok in name.lower() for tok in bad):
            continue
        out.append(name)
    return out

def _parse_issued(issued):
    dp = (issued or {}).get("date-parts", [[]])
    ymd = dp[0] if dp and dp[0] else []
    return "-".join(str(x) for x in ymd) if ymd else None

def _clean_abstract(abstract_raw):
    if not abstract_raw:
        return None
    text = re.sub("<[^>]+>", " ", abstract_raw)  # remove JATS/HTML
    return html.unescape(" ".join(text.split())) or None

# Sanitize topic (strip code fences / ReAct noise / accidental JSON)
def _sanitize_topic(topic: str) -> str:
    t = (topic or "").strip().replace("```", "")
    # if the model sent JSON as a string, extract the real topic
    if t.startswith("{") and '"topic"' in t:
        try:
            obj = json.loads(t)
            if isinstance(obj, dict) and isinstance(obj.get("topic"), str):
                t = obj["topic"].strip()
        except Exception:
            pass
    # cut off typical ReAct artefacts
    for cut in ["\nObserv", "\nObservation:", "\nFinal Answer:", "\nThought:", "\nAction Input:"]:
        idx = t.find(cut)
        if idx != -1:
            t = t[:idx].strip()
    return t[:200]

def crossref_search_latest_clean(topic: str, rows: int = 5,
                                 from_date: Optional[str] = None,
                                 until_date: Optional[str] = None,
                                 require_abstract: bool = True) -> List[Dict]:
    today = date.today()
    if not until_date:
        until_date = today.isoformat()
    if not from_date:
        from_date = f"{today.year-1}-01-01"

    topic = _sanitize_topic(topic)

    filters = [
        f"from-pub-date:{from_date}",
        f"until-pub-date:{until_date}",
        "type:journal-article",
    ]
    if require_abstract:
        filters.append("has-abstract:true")

    params = {
        "query": topic,
        "sort": "published",
        "order": "desc",
        "select": "DOI,title,author,issued,abstract,URL",
        "filter": ",".join(filters),
        "mailto": "veapasichnyk@gmail.com",
    }

    # network/HTTP-safe call
    resp = None
    try:
        resp = requests.get(CROSSREF_ENDPOINT, params=params, timeout=20)
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        r = e.response
        status = getattr(r, "status_code", "unknown")
        body = getattr(r, "text", "")[:300] if r is not None else str(e)
        raise ValueError(f"Crossref HTTP {status}: {body}") from e
    except requests.exceptions.RequestException as e:
        raise ValueError(f"Crossref request failed: {e}") from e

    items = resp.json().get("message", {}).get("items", [])

    out, seen = [], set()
    current_year = today.year
    for it in items:
        title = (it.get("title") or [""])[0].strip()
        if not title or title.lower().startswith("title pending"):
            continue

        issued = _parse_issued(it.get("issued"))
        if issued:
            try:
                y = int(issued.split("-")[0])
                if y > current_year or y < 1900:
                    continue
            except:
                pass

        doi, url = it.get("DOI"), it.get("URL")
        key = doi or url or title.lower()
        if key in seen:
            continue
        seen.add(key)

        out.append({
            "title": title,
            "authors": _format_authors(it.get("author", [])),
            "issued": issued,
            "doi": f"https://doi.org/{doi}" if doi else None,
            "url": url,
            "abstract": _clean_abstract(it.get("abstract")) or "No abstract"
        })
        if len(out) >= rows:
            break
    return out

CrossrefTool = StructuredTool.from_function(
    name="crossref_search_latest_clean",
    description="Пошук останніх journal-article у Crossref за темою. Видає title, authors, issued, doi/url, abstract.",
    func=crossref_search_latest_clean,
    args_schema=CrossrefSearchInput,
)

In [31]:
#DuckDuckGo Tool
from langchain_community.tools import DuckDuckGoSearchResults

DDGTool = DuckDuckGoSearchResults(
    name="duckduckgo_search",
    description="Веб-пошук (новини або текст) для уточнення теми/підтем.",
    backend="news"
)

In [37]:
#Prompt
from langchain_core.prompts import PromptTemplate

RESEARCH_PROMPT = PromptTemplate(
    input_variables=["input", "agent_scratchpad", "tools", "tool_names", "k"],
    template="""
You are a meticulous ReAct research agent.

STRICT Tool-call rules:
- Action Input MUST be a VALID JSON object (no markdown fences, no extra text).
- For `crossref_search_latest_clean` the JSON MUST look like:
  {{"topic": "<short query>", "rows": <int>, "require_abstract": true}}
- NEVER wrap JSON in code fences. No commentary inside JSON.

Output requirements (for each of the {k} items):
- Title: exact paper title
- Authors: comma-separated names only (no affiliations)
- Date: publication date (YYYY or YYYY-MM-DD if available)
- Description: concise 2–3 sentence summary (or 'No abstract')

ReAct protocol (follow EXACTLY):
Question: {input}
Thought: brief plan
Action: one of [{tool_names}]
Action Input: {{ ...valid JSON... }}
Observation: tool result
... (repeat if needed)
Thought: I have enough information
Final Answer:
1) Title: ...
   Authors: ...
   Date: ...
   Description: ...
2) Title: ...
   Authors: ...
   Date: ...
   Description: ...
(continue until exactly {k} items)

HARD STOP RULES:
- After you write the section that begins with the exact token "Final Answer:", DO NOT write any more Thought/Action/Observation. STOP OUTPUT COMPLETELY.

Available tools: {tool_names}

TOOLS:
{tools}

{agent_scratchpad}
""".strip()
)

In [38]:
#Створення ReAct-агента + LLM (Hugging Face)
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import Sequence
from langchain_core.tools import BaseTool
from langchain.agents import AgentExecutor
from langchain.agents.react.agent import create_react_agent

creds = load_creds()
token = creds.get("hf_api_token")

# Базовий ендпоінт у режимі conversational (сумісний з провайдерами)
base_llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    task="conversational",
    temperature=0,
    huggingfacehub_api_token=token
)

# Чат-обгортка, яку і передаємо агенту
llm = ChatHuggingFace(llm=base_llm)


# Cтворюємо агента
tools = [CrossrefTool, DDGTool]
agent = create_react_agent(llm=llm, tools=tools, prompt=RESEARCH_PROMPT)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors="Follow the specified ReAct format or stop after Final Answer.",
    max_iterations=8,
    early_stopping_method="force"
)

In [39]:
query = "Find the 5 most recent publications about large language models"
k = 5
result = agent_executor.invoke({"input": query, "k": k})
print(result["output"])



> Entering new AgentExecutor chain...
 Thought: I will use the `crossref_search_latest_clean` tool to find the 5 most recent publications about large language models.

Action: crossref_search_latest_clean
Action Input: {"topic": "large language models", "rows": 5}

Observ[{'title': 'Inner-character and Inner-word Features Based Representation Learning for Chinese Word Embedding', 'authors': ['Yun Zhang', 'Yongguo Liu', 'Jiajing Zhu', 'Zhi Chen', 'Shuangqing Zhai', 'Xindong Wu'], 'issued': '2025-9-11', 'doi': 'https://doi.org/10.1145/3748316', 'url': 'https://doi.org/10.1145/3748316', 'abstract': 'Chinese word embedding is a significant task in natural language processing (NLP). Most researchers explored Chinese word embedding according to radical, component, stroke n -gram and character features. Besides these features, Chinese characters still have structure and pinyin characteristics. In this article, we propose ensemble ssp2vec and connective ssp2vec to utilize inner-character fea



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [22]:
!pip install -q -U langchain langchain-openai langchain-experimental pydantic requests python-dateutil
!pip install -q -U langchain-tavily


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.9/444.9 kB 10.1 MB/s eta 0:00:00


In [15]:
import json, getpass

openai_token = getpass.getpass("Введи свій OPENAI_API_KEY токен: ")
tvly_token = getpass.getpass("Введи свій TAVILY_API_KEY токен: ")

creds = {
    "OPENAI_API_KEY": openai_token,
    "TAVILY_API_KEY": tvly_token
}

with open("creds.json", "w") as f:
    json.dump(creds, f)

print("creds.json створено")

Введи свій OPENAI_API_KEY токен: ··········
Введи свій TAVILY_API_KEY токен: ··········
creds.json створено


In [16]:
# Завантажуємо ключі назад і встановлюємо в середовище
import os
with open("creds.json", "r") as f:
    creds_loaded = json.load(f)

os.environ["OPENAI_API_KEY"] = creds_loaded["OPENAI_API_KEY"]
os.environ["TAVILY_API_KEY"] = creds_loaded["TAVILY_API_KEY"]

print("Ключі збережені в os.environ")

Ключі збережені в os.environ


In [30]:
# Imports
import re
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.tools.python.tool import PythonREPLTool

In [24]:
# Utilities
YEAR_RE = re.compile(r"(20\d{2})\D+(\d+(?:[.,]\d+)?)", re.I)

def parse_year_series(text: str) -> List[Tuple[int, float]]:
    pairs = []
    for m in YEAR_RE.finditer(text):
        y = int(m.group(1))
        v = float(m.group(2).replace(",", "."))
        pairs.append((y, v))
    return sorted(pairs, key=lambda x: x[0])

@dataclass
class ExternalSignals:
    inflation_yoy: Optional[float] = None
    demand_index: Optional[float] = None
    weather_factor: Optional[float] = None

class ForecastResult(BaseModel):
    forecast_year: int
    point_forecast_tons: float
    range_low_tons: float
    range_high_tons: float
    baseline_growth: float
    inflation_adj: float
    demand_adj: float
    weather_adj: float
    notes: str
    sources: List[str]

def compute_baseline_growth(series: List[Tuple[int, float]]) -> float:
    if len(series) < 2: return 0.0
    y0, v0 = series[0]; y1, v1 = series[-1]
    years = max(1, y1 - y0)
    return (v1 / v0) ** (1 / years) - 1 if v0 > 0 else 0.0

def apply_adjustments(last_value: float, baseline_g: float, signals: ExternalSignals):
    # прості ліміти/ваги
    inflation_adj = -0.4 * signals.inflation_yoy if signals.inflation_yoy else 0.0
    demand_adj = 0.08 * (signals.demand_index - 1.0) if signals.demand_index else 0.0
    weather_adj = signals.weather_factor if signals.weather_factor else 0.0
    total_growth = max(-0.2, min(0.2, baseline_g + inflation_adj + demand_adj + weather_adj))
    point = last_value * (1 + total_growth)
    band = max(0.02, 0.5 * (abs(inflation_adj) + abs(demand_adj) + abs(weather_adj)))
    return point, {"inflation_adj": inflation_adj, "demand_adj": demand_adj,
                   "weather_adj": weather_adj, "band": band, "total_growth": total_growth}

In [25]:
# Tools
from langchain.tools import tool

@tool("parse_series")
def parse_series_tool(text: str) -> str:
    """Extract (year, tons) pairs from user text.
    Input: free text with years and numeric volumes (e.g., '2023 - 190т').
    Output: JSON list: [{"year": int, "tons": float}, ...].
    """
    pairs = parse_year_series(text)
    return json.dumps([{"year": y, "tons": v} for y, v in pairs], ensure_ascii=False)

@tool("calc_forecast")
def calc_forecast(json_payload: str) -> str:
    """Compute a 1-year-ahead forecast from JSON.
    Expects keys:
      - series: list of {"year": int, "tons": float}
      - forecast_year: int (optional, defaults to last_year + 1)
      - inflation_yoy: float (optional, e.g., 0.045 for +4.5%)
      - demand_index: float (optional, 1.0 = neutral)
      - weather_factor: float (optional, e.g., -0.02)
      - sources: list[str] (optional)
    Returns: JSON matching ForecastResult.
    """
    try:
        data = json.loads(json_payload)
    except Exception as e:
        return json.dumps({"error": f"Invalid JSON: {e}"}, ensure_ascii=False)

    series_raw = data.get("series", [])
    if not series_raw:
        return json.dumps({"error": "Missing 'series' with year/tons pairs."}, ensure_ascii=False)

    try:
        series = [(int(d["year"]), float(d["tons"])) for d in series_raw]
    except Exception as e:
        return json.dumps({"error": f"Bad 'series' format: {e}"}, ensure_ascii=False)

    series.sort(key=lambda x: x[0])
    last_year, last_value = series[-1]
    forecast_year = int(data.get("forecast_year", last_year + 1))

    signals = ExternalSignals(
        inflation_yoy=(float(data["inflation_yoy"]) if data.get("inflation_yoy") is not None else None),
        demand_index=(float(data["demand_index"]) if data.get("demand_index") is not None else None),
        weather_factor=(float(data["weather_factor"]) if data.get("weather_factor") is not None else None),
    )

    baseline_g = compute_baseline_growth(series)
    point, meta = apply_adjustments(last_value, baseline_g, signals)

    result = ForecastResult(
        forecast_year=forecast_year,
        point_forecast_tons=round(point, 2),
        range_low_tons=round(point*(1-meta["band"]), 2),
        range_high_tons=round(point*(1+meta["band"]), 2),
        baseline_growth=round(baseline_g, 4),
        inflation_adj=round(meta["inflation_adj"], 4),
        demand_adj=round(meta["demand_adj"], 4),
        weather_adj=round(meta["weather_adj"], 4),
        notes="Baseline uses CAGR; simple caps on adjustments.",
        sources=data.get("sources", []),
    )
    return result.model_dump_json()

In [26]:
# External tools (updated Tavily)
search_tool = TavilySearch(max_results=5)  # from langchain_tavily
python_tool = PythonREPLTool()

In [34]:
from langchain_core.tools import render_text_description

SYSTEM = """You are a Business Analytics Assistant.
You have access to tools:
{tools}

Your task: build a 1-year-forward forecast using baseline trend + adjustments (inflation, demand, weather).
Return executive summary, assumptions, table, sources, and data JSON.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{user_request}"),
    MessagesPlaceholder("agent_scratchpad"),
])

tools = [parse_series_tool, search_tool, python_tool, calc_forecast]
tool_desc = render_text_description(tools)
prompt = prompt.partial(tools=tool_desc)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
agent = create_openai_tools_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [40]:
request = """
Ми експортуємо апельсини з Бразилії.
У 2022 експортували 200т, у 2023 — 190т, у 2024 — 210т, у 2025 (ще не закінчився) — 220т.
Зроби оцінку на 2026, враховуючи інфляцію та погодні умови.
"""
out = executor.invoke({"user_request": request})
print(out["output"][:2000])



> Entering new AgentExecutor chain...

Invoking: `parse_series` with `{'text': '2022 - 200т, 2023 - 190т, 2024 - 210т, 2025 - 220т'}`


[{"year": 2022, "tons": 200.0}, {"year": 2023, "tons": 190.0}, {"year": 2024, "tons": 210.0}, {"year": 2025, "tons": 220.0}]
Invoking: `calc_forecast` with `{'json_payload': '{"series":[{"year":2022,"tons":200},{"year":2023,"tons":190},{"year":2024,"tons":210},{"year":2025,"tons":220}],"forecast_year":2026,"inflation_yoy":0.045,"demand_index":1.0,"weather_factor":-0.02,"sources":[]}'}`


{"forecast_year":2026,"point_forecast_tons":218.74,"range_low_tons":214.37,"range_high_tons":223.12,"baseline_growth":0.0323,"inflation_adj":-0.018,"demand_adj":0.0,"weather_adj":-0.02,"notes":"Baseline uses CAGR; simple caps on adjustments.","sources":[]}### Executive Summary
The forecast for orange exports from Brazil in 2026 is estimated at approximately **218.74 tons**. This estimate takes into account the baseline growth trend, inflation, demand conditions, and 

In [41]:
#З акцентом на веб-пошук (щоб агент точно підтягнув дані)
request = """
Ми експортуємо мед з України.
У 2022 експортували 60т, у 2023 — 55т, у 2024 — 62т, у 2025 (ще не закінчився) — 64т.
Зроби оцінку на 2026, обов’язково використай актуальні дані про інфляцію в Україні (2024–2025)
та сезонний кліматичний прогноз (Copernicus/ECMWF/meteo.gov.ua) як погодний фактор.
Додай джерела. Якщо бракує даних — чітко вкажи, чого саме не вистачає, без вигаданих чисел.
"""
out = executor.invoke({"user_request": request})

print(out["output"][:2000])



> Entering new AgentExecutor chain...

Invoking: `parse_series` with `{'text': '2022 - 60т, 2023 - 55т, 2024 - 62т, 2025 - 64т'}`


[{"year": 2022, "tons": 60.0}, {"year": 2023, "tons": 55.0}, {"year": 2024, "tons": 62.0}, {"year": 2025, "tons": 64.0}]
Invoking: `tavily_search` with `{'query': 'Ukraine inflation rates 2024 2025', 'search_depth': 'advanced', 'topic': 'finance'}`


{'query': 'Ukraine inflation rates 2024 2025', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://finance.yahoo.com/news/ukraine-inflation-set-peak-mid-164859204.html', 'title': "Ukraine's inflation set to peak by mid-2025, cool to 8% by year ...", 'content': "Ukraine's inflation rate is expected to peak at 15% by mid-2025 before dropping to 8.4% by year's end, Ukraine's Central Bank said on Jan.", 'score': 0.89849097, 'raw_content': None}, {'url': 'https://uk.finance.yahoo.com/news/ukraine-holds-key-rate-13-152502785.html', 'title': 'Ukraine holds key rate at 13% as infl

**Підсумки**

На мій погляд агент правильно зпрогнозував сценарій для меду з України:

- Він витягнув історію експорту (2022–2025).

- Підтягнув інфляцію.

- Врахував погодні умови (знайдено ризики суворої зими, дав −0.02).

- Використав нейтральний попит (demand_index = 1.0).

- Видав прогноз на 2026 рік: ~61.12 т (діапазон 59.08–63.16 т).

- Чітко вказав джерела (Yahoo Finance, WHO winter risk report).

Агент також зазначив, що бракує більш детальних прогнозів погоди саме для сезону медозбору 2026 року - це логічно, бо поки доступні лише середньострокові оцінки.

**Пропозиції для покращення**

- Можна зробити «one-step-ahead» бектест: тренуйся на 2022–t-1, прогнозуй t (t=2023..2025), рахуй MAPE/RMSE. Це дасть довіру до підходу.

- Локалізація: дублювати вихід українською/англійською, щоб одразу можна було вставляти результати у звіт.